# Research workflow — explicit Bitfinex book capture

Owner: repository founder. Current workflow: 2026-09-23.

This maintained live journey uses the supported Bitfinex public R0 book route. It is safe to open and run without a venue call: the default mode only reports that capture is skipped. Set `ALPHA_QR_CAPTURE=1` to opt in to one new bounded public recording, or set `ALPHA_BITFINEX_CAPTURE_DIRECTORY` to reopen an existing recording.

The notebook retains the selected recording, publishes an explicit 32-position prefix, reopens it through the shared research reader and computes a visible-size imbalance for inspection. It does not use the deferred Binance/Tardis P1 investigation. The [historical P1 notebook](research-foundations-live-p1-evidence.ipynb) remains available with its original source and outputs identity.

[Offline research](research-foundations.ipynb) · [Run and retain](notebook-operations.md#live-versions) · [Bitfinex capture guide](../docs/data/l3-collectors.md#start-here)

## 1. Select the research question

The question is deliberately bounded: what visible-size imbalance is observable in a received Bitfinex R0 order window? This is exploratory book evidence with no forward label, parameter search, economic acceptance or execution claim.

In [ ]:
# standard library imports
from datetime import UTC, datetime, timedelta
from fractions import Fraction
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
from uuid import uuid4

# third party library imports
import pandas as pd
from IPython.display import display

# project library imports
from configuration import CaptureConfig
from market_data.adapters.sources import BookAdapters, SourceAdapters
from market_data.contracts import CaptureRequest, Order
from market_data.datasets import DatasetPublication
from market_data.query import BookMarketRow, BookQuery
from market_data.replay import SavedCapture
from market_data.storage import RawStore
from research.exploration import (
    BookResearchInput,
    BookSelection,
    ResearchDataset,
    ResearchReader,
    ResearchSelection,
)
from runtime import ImplementationIdentity
from serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/live/bitfinex-btc-usd.json").is_file(), "Use the repository root/kernel"
assert platform.python_version() == (root / ".python-version").read_text().strip()
codec = DomainCodec()
pd.set_option("display.max_colwidth", None)
workflow = os.environ.get("ALPHA_QR_WORKFLOW", "book-capture")
if workflow != "book-capture":
    raise ValueError("The maintained live notebook supports only the Bitfinex book-capture workflow")
capture_requested = os.environ.get("ALPHA_QR_CAPTURE", "").lower() in {"1", "true", "yes"}
saved_capture = os.environ.get("ALPHA_BITFINEX_CAPTURE_DIRECTORY", "")
if capture_requested and saved_capture:
    raise ValueError("Choose a new capture or an existing recording, not both")
question = {
    "question": "What visible-size imbalance is observable in this bounded received order window?",
    "source": "public Bitfinex R0",
    "comparator": "zero imbalance; no performance claim",
    "fitting": "not applicable",
    "holdout": "not opened",
}
print("Selected workflow:", workflow)
print("External capture explicitly enabled:", capture_requested)
question

## 2. Prepare an explicit saved-input or capture attempt

The capture cell below is the only cell that can contact Bitfinex, and it does so only when `ALPHA_QR_CAPTURE=1`. A supplied `ALPHA_BITFINEX_CAPTURE_DIRECTORY` is read locally. Every active attempt gets a new ignored directory; source bytes and logs stay outside Git.

In [ ]:
active = False
study = None
capture_directory = None
config = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
if config.venue != "bitfinex" or config.channel != "R0" or config.duration_seconds > 20:
    raise ValueError("This notebook selects public Bitfinex R0 with a receive interval <=20s")
if capture_requested:
    active = True
    study = root / "data/notebooks/research-live" / str(uuid4())
    study.mkdir(parents=True)
    (study / "raw").mkdir()
    (study / "snapshots").mkdir()
    config_path = study / "capture-config.json"
    config_path.write_bytes(codec.encode_json(config))
    (study / "attempt.json").write_bytes(codec.encode_json({"question": question, "identity": ImplementationIdentity.sha256()}))
    collector = Path(sys.executable).parent / "alpha-collect"
    if not collector.is_file():
        raise FileNotFoundError(f"Installed collector is missing: {collector}")
    command = [
        str(collector),
        "capture",
        "--config",
        str(config_path),
        "--output",
        str(study / "captures"),
        "--snapshot-directory",
        str(study / "snapshots"),
        "--snapshot-records",
        "32",
    ]
    (study / "command.json").write_bytes(codec.encode_json(command))
    with (study / "capture.stdout").open("x") as stdout, (study / "capture.stderr").open("x") as stderr:
        capture = subprocess.run(command, cwd=study, stdout=stdout, stderr=stderr, timeout=80, check=False)
    (study / "capture-exit.json").write_bytes(codec.encode_json({"exit_code": capture.returncode}))
    if capture.returncode != 0:
        raise RuntimeError(f"Capture exit {capture.returncode}; inspect retained logs and prefix under {study}")
    summary = json.loads((study / "capture.stdout").read_text())
    capture_directory = Path(summary["capture_directory"]).resolve()
    request = codec.decode_json((capture_directory / "request.json").read_bytes(), CaptureRequest)
    assert request.config == config and request.identity == summary["request_identity"]
elif saved_capture:
    active = True
    capture_directory = Path(saved_capture).expanduser().resolve()
    if not capture_directory.is_dir():
        raise FileNotFoundError(f"Selected Bitfinex recording is missing: {capture_directory}")
    request = codec.decode_json((capture_directory / "request.json").read_bytes(), CaptureRequest)
    if request.config != config:
        raise ValueError("Selected recording does not match configs/live/bitfinex-btc-usd.json")
    study = root / "data/notebooks/research-live" / str(uuid4())
    study.mkdir(parents=True)
    (study / "raw").mkdir()
    (study / "snapshots").mkdir()
    (study / "attempt.json").write_bytes(codec.encode_json({"question": question, "identity": ImplementationIdentity.sha256(), "source": str(capture_directory)}))
else:
    print("SKIPPED: no capture selected. Set ALPHA_BITFINEX_CAPTURE_DIRECTORY to reopen a recording, or ALPHA_QR_CAPTURE=1 to opt in to one public capture.")

## 3. Reopen, publish and inspect the pinned book prefix

Publication and input selection are explicit operations after a completed recording. The reader preserves control boundaries and does not infer an empty market from missing states.

In [ ]:
if active:
    replay = SavedCapture.read(capture_directory, book_factory=BookAdapters.create)
    summary = replay.summary()
    if not summary["ended"]:
        raise ValueError("Selected recording is not a completed capture; use the engineer workflow for explicit interrupted-prefix inspection")
    snapshot_id = DatasetPublication(SourceAdapters(), ImplementationIdentity.sha256()).publish_books(
        (capture_directory,),
        RawStore(study / "raw"),
        study / "snapshots",
        max_records=32,
        max_bytes=config.max_bytes,
    )
    if summary["started_ns"] is None or summary["finished_ns"] is None:
        raise ValueError("Completed recording has no receipt interval")
    epoch = datetime(1970, 1, 1, tzinfo=UTC)
    start = epoch + timedelta(microseconds=summary["started_ns"] // 1000)
    end = epoch + timedelta(microseconds=summary["finished_ns"] // 1000 + 1)
    selection = ResearchSelection(
        inputs=(
            BookSelection(
                name="visible_depth",
                query=BookQuery(
                    snapshot_id=snapshot_id,
                    request_identity=request.identity,
                    instrument=config.instrument_id,
                    start=start,
                    end=end,
                    known_at=end,
                    columns=("orders",),
                    batch_size=32,
                    max_bytes=config.max_bytes,
                ),
            ),
        )
    )
    selection_path = study / "selection.json"
    selection_path.write_bytes(codec.encode_json(selection))
    dataset = ResearchReader(study / "snapshots", RawStore(study / "raw"), readers=SourceAdapters()).read(selection)
    assert isinstance(dataset, ResearchDataset) and dataset.identity == selection.identity
    depth = dataset.inputs[0]
    assert isinstance(depth, BookResearchInput)
    rows = [row for batch in depth.batches for row in batch.rows]
    market_rows = [row for row in rows if isinstance(row, BookMarketRow)]
    controls = [row for row in rows if not isinstance(row, BookMarketRow)]
    display(pd.DataFrame([{"market_states": len(market_rows), "control_boundaries": len(controls), "receipt_start": start, "receipt_end": end, "quality": sorted({reason for batch in depth.batches for reason in batch.reasons})}]))
    display(pd.DataFrame([{"position": row.position, "received_ns": row.received_ns, "status": row.status, "reason": row.reason} for row in controls]))
else:
    print("No saved selection to inspect.")

## 4. Compute a visible-size diagnostic

For each admitted received book state, sum exact base quantities on each side and retain the dimensionless imbalance. This uses only message-local visible orders; it does not forward-fill controls, infer missing depth or create a return label.

In [ ]:
if active:
    prototype = []
    for row in market_rows:
        orders = row.values[0]
        assert all(isinstance(order, Order) for order in orders)
        bids = sum((Fraction(order.size) for order in orders if order.side == "buy"), Fraction())
        asks = sum((Fraction(order.size) for order in orders if order.side == "sell"), Fraction())
        total = bids + asks
        imbalance = None if total == 0 else (bids - asks) / total
        prototype.append({"position": row.position, "received_ns": row.received_ns, "state": row.state_identity, "quality": row.status, "imbalance_exact_ratio": None if imbalance is None else str(imbalance)})
    (study / "prototype.json").write_bytes(codec.encode_json(prototype))
    print("Computed exploratory states:", len(prototype))
    print("Unavailable denominators:", sum(item["imbalance_exact_ratio"] is None for item in prototype))
    print("Retained exact ratios and state identities:", study / "prototype.json")
else:
    print("No active book states; diagnostic skipped.")

## 5. Handoff

The retained recording and selection can be reopened by the offline QR journey or the engineer diagnostics. This exploratory Bitfinex route does not qualify historical coverage, close P1, establish profitability or authorize execution.

In [ ]:
if active:
    handoff = {"selection": selection.identity, "snapshot": snapshot_id, "request": request.identity, "capture": str(capture_directory), "market_states": len(market_rows), "status": "exploratory; no economic acceptance"}
    (study / "handoff.json").write_bytes(codec.encode_json(handoff))
    print("Offline QR: set ALPHA_RESEARCH_DATASET_DIRECTORY to", study)
    print("Offline engineer: set ALPHA_BITFINEX_CAPTURE_DIRECTORY to", capture_directory)
    print("Retained all selected inputs and attempts; no automatic deletion or repeated capture.")
else:
    print("No handoff was created because no capture or saved recording was selected.")